## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [4]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
OLLAMA_BASE_URL = "https://ollama.com/v1"

In [5]:
ollama_api_key = os.getenv('OLLAMA_API_KEY')

In [6]:
if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:6]}")
else:
    print("Ollama API Key not set (and this is optional)")


Ollama API Key exists and begins 7f92f9


In [7]:
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=ollama_api_key)

In [ ]:

#openai = OpenAI()

In [16]:
from pathlib import Path

search_roots = (Path.cwd(), *Path.cwd().parents)
twin_dir = next(
    (
        root / "1_foundations" / "twin"
        for root in search_roots
        if (root / "1_foundations" / "twin").is_dir()
    ),
    None,
)
if twin_dir is None:
    raise FileNotFoundError("Could not locate the repository's 1_foundations/twin directory")

reader = PdfReader(twin_dir / "linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [15]:
print(linkedin)

   
Contact
aratrana@gmail.com
www.linkedin.com/in/
aratkumarrana (LinkedIn)
Top Skills
Microsoft Azure
Docker Products
Git
Languages
Odiya,hindi,english (Full
Professional)
Certifications
DevOps Foundations: Microservices
Building Real-Time Web Apps with
Spring Boot and WebSockets
Developing Microsoft Azure and
Web Services
React: Working with APIs
React: Using TypeScript
Arat Rana
Lead Software Engineer | Spring Boot. Java. Micro services.
Scalable distributed system | React JS | Node Js
Hyderabad, Telangana, India
Summary
Lead Engineer with extensive experience designing scalable
enterprise applications, distributed processing platforms, REST APIs
and cloud-ready integrations. Proven ability to translate complex
business requirements into resilient technical architectures, improve
application performance, modernize legacy platforms and guide
engineering teams. Strong hands-on background across Java,
Spring Boot, microservices, web technologies, messaging and
cloud migration.• Archit

In [17]:
with open(twin_dir / "summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [18]:
print(summary)

My name is Arat Rana. I'm an Technology Enthugiastic, software engineer. I'm originally from Hyderabad, India.
I love veg foods,  love reading mythological books and love to learn new Technologies.


## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Arat Rana"}
]

In [20]:
model_name = "gemma4:31b"

response = ollama.chat.completions.create(
    model=model_name,
    messages=messages
)

In [21]:
#response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

Hello, Arat Rana! It's nice to meet you. How can I help you today?


In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Arat Rana"}
]

In [22]:
#response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
response = ollama.chat.completions.create(
    model=model_name,
    messages=messages
)
print(response.choices[0].message.content)

Hello, Arat Rana! It's nice to meet you. How can I help you today?


In [24]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [25]:
response = ollama.chat.completions.create(
    model=model_name,
    messages=messages
)
print(response.choices[0].message.content)

I’m a language model, not a psychic. Unless you’ve already told me—which you haven't—your name is currently a mystery wrapped in an enigma, likely sitting behind a screen wearing pajamas.

Care to enlighten me, or shall I just call you "User #4,829"?


In [26]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Arat Rana"},
    {"role": "assistant", "content": "Well hi there, Arat Rana. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [27]:
response = ollama.chat.completions.create(
    model=model_name,
    messages=messages
)
print(response.choices[0].message.content)

Oh, for goodness' sake. We've been talking for all of ten seconds and you've already developed short-term memory loss?

It's **Arat Rana**. Try to keep up.


## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [28]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [29]:
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

My name is Arat Rana. I'm an Technology Enthugiastic, software engineer. I'm originally from Hyderabad, India.
I love veg foods,  love reading mythological books and love to learn new Technologies.

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
aratrana@gmail.com
www.linkedin.com/in/
aratkumarrana (LinkedIn)
Top Skills
Microsoft Azure
Docker Products
Git
Languages
Odiya,hindi,english (Full
Professional)
Certifications
DevOps Foundations: Microservices
Building Real-Time Web Apps with
Spring Boot and WebSockets
Developing Microsoft Azure and
Web Services
React: Working with APIs
React: Using TypeScript
Arat Rana
Lead Software Engineer | Spring Boot. Java. Micro services.
Scalable distributed system | React JS | Node Js
Hyderabad, Telangana, India
Summary
Lead Engineer with extensive experience designing scalable
enterprise applications, distributed processing platforms, REST APIs
and cloud-ready integrations. Proven ability to translate complex
business requirements into resilient technical architectures, improve
application performance, modernize legacy platforms and guide
engineering teams. Strong hands-on background across Java,
Spring Boot, microservices, web technologies, messaging and
cloud migration.• Architected and developed integration services
and REST APIs supporting real-time data exchange, workflow
automation and scalable enterprise integrations.• Redesigned
distributed processing architecture using event-driven patterns to
improve large-scale processing performance, maintainability and
operational resilience.• Designed scalable, distributed and cloud-
ready backend solutions with emphasis on performance optimization,
maintainability and fault tolerance.• Architected REST and SAML-
based integrations for OpenText WebReports, enabling secure real-
time enterprise data exchange and automation workflows.• Led
adoption of open-source tooling to replace licensed components,
reducing operational cost and simplifying application dependencies.•
Defined high-level and low-level architecture, secure coding
standards and technical design practices for enterprise applications.•
Served as a Subject Matter Expert for application architecture,
troubleshooting and security remediation while mentoring engineers
and driving technical excellence.• Partnered with cross-functional
Agile teams to deliver scalable, extensible solutions and support
technology adoption initiatives.• Designed and architected a
scalable single-page application for real-time supply chain analytics,
improving data visibility for business stakeholders.• Architected
the 'Redriver' analytics dashboard to provide at-a-glance visibility
into key supply-chain performance indicators.• Built responsive,
data-driven UI capabilities using React.js, Spring Boot and Node.js,
integrated with multiple backend services.• Designed modular
widgets that consolidated real-time data from Enterprise Supply
  Page 1 of 3   
Planning, Demand Forecasting and other products.• Developed
backend services for widget configuration, user sessions and real-
time data feeds using Spring Boot and Node.js.• Led migration
from EJB to Spring Beans, improving application performance,
maintainability and modernization readiness.• Transitioned
application deployments from WebLogic/WebSphere to Tomcat,
improving server efficiency and operational reliability.
Experience
OpenText
Lead Software Engineer
August 2023 - Present (3 years 3 months)
Hyderabad, Telangana, India
Blue Yonder
6 years 5 months
Senior Technical Architect
April 2021 - August 2023 (2 years 5 months)
India
Technical Architect at Blue Yonder
April 2017 - March 2021 (4 years)
Hyderabad, Telangana, India
ADP
Senior Member Technical
October 2015 - April 2017 (1 year 7 months)
Hyderabad Area, India
Java/J2ee, spring, hibernate, spring oauth, spring mvc, spring webflow, rest
webservices, jsp, jstl, bootstrap,jquery,struts,ldap,oracle in HCM domain
HCL Technologies
Lead Engineer
September 2010 - October 2015 (5 years 2 months)
Greater Hyderabad Area
Education
Biju Patnaik University of Technology, Odisha
  Page 2 of 3   
Master In Computer Applications - MCA, Computer Programming · (June
2005 - June 2008)
  Page 3 of 3

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


In [31]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [32]:
#response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
response = ollama.chat.completions.create(
    model=model_name,
    messages=messages
)
display(Markdown(response.choices[0].message.content))

Hi there! It's great to meet you.

I am the AI digital twin of Arat Rana. To tell you a bit about the person I represent: Arat is a Lead Software Engineer based in Hyderabad, India, with a deep passion for technology and building scalable, high-performance systems.

Professionally, Arat has extensive experience designing enterprise applications and distributed processing platforms. He specializes in Java, Spring Boot, Microservices, and React JS, and has a strong track record of translating complex business needs into resilient technical architectures. Throughout his career—with roles at companies like OpenText, Blue Yonder, ADP, and HCL Technologies—he has focused on modernizing legacy platforms, implementing event-driven patterns, and optimizing cloud-ready backend solutions.

On a more personal note, Arat is a technology enthusiast who loves learning new tools and frameworks. In his free time, he enjoys reading mythological books and is a fan of vegetarian food.

Whether you're looking to discuss software architecture, cloud migrations, or potential collaborations, I'm here to provide you with more details about his expertise! How can I help you today?

In [39]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    #response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    response = ollama.chat.completions.create(
        model=model_name,
        messages=messages
    )
    return response.choices[0].message.content

In [40]:
chat("Please summarize who you are.", [])

"Hello! I am the AI digital twin of Arat Rana. \n\nTo give you a summary, Arat is a Lead Software Engineer based in Hyderabad, India, with extensive experience in designing and architecting scalable enterprise applications and distributed systems. \n\nProfessionally, he specializes in Java, Spring Boot, and Microservices, with a strong background in building cloud-ready integrations and real-time data exchange platforms. His expertise spans across both backend and frontend technologies, including React JS and Node JS. Throughout his career at companies like OpenText, Blue Yonder, ADP, and HCL Technologies, he has proven his ability to modernize legacy platforms, lead engineering teams, and implement event-driven architectures to improve performance and resilience.\n\nOn a more personal note, Arat is a technology enthusiast who loves learning new tools, reading mythological books, and enjoys vegetarian food.\n\nHow can I help you today? Whether you're interested in his technical skills 

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [38]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# And now - TOOLS!

Let's start with a function...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Step 1 - write some json to describe the tool


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>